In [1]:
library(readxl)
library(tidyverse)
library(igraph)
library(writexl)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Adjuntando el paquete: 'igraph'


The following objects are masked from 'package:lubridate':

    %--%, union


The following objects are masked from 'package:dplyr':

    as_data_frame, groups, union


The following objects are masked from 'package:purrr':

    compose, simplify


The following object is masked from 'package:tidyr':

    crossing


The following object is masked from 'package:tibble':

    as_data_frame


The following objects are masked from 'package:s

In [2]:
path <- "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Raw_Data\\Matriz_de_Adyacencia_Pensiones_v6.xlsx"
df <- read_excel(path, sheet = "Hoja2", col_names = TRUE)

New names:
• `` -> `...1`


In [3]:
A <- df %>%
  rename(node = 1) %>%                            # primera columna = nombres fila
  mutate(across(-node, ~replace_na(as.numeric(.x), 0))) %>%  # NA->0, a numérico
  column_to_rownames("node") %>%
  as.matrix()

In [4]:
stopifnot(nrow(A) == ncol(A))
stopifnot(all(A %in% c(0,1)))

# Autodependencias (i depende de sí mismo)
self_loops <- rownames(A)[diag(A) == 1]
self_loops

character(0)

In [5]:
## Generación del grafo sin autodependencias
A_nodiag <- A
diag(A_nodiag) <- 0

g <- graph_from_adjacency_matrix(A_nodiag, mode = "directed", diag = FALSE)

c(vcount(g), ecount(g))  # (n_nodos, n_arcos)

[1] 36 96

In [6]:
## Dependencias mutuas (i depende de j y j depende de i)

mutual_idx <- which(A_nodiag == 1 & t(A_nodiag) == 1, arr.ind = TRUE)

direct_codep <- if (nrow(mutual_idx) == 0) {
  tibble(a = character(), b = character())
} else {
  pairs <- unique(t(apply(mutual_idx, 1, function(rc) sort(rc))))
  tibble(a = rownames(A_nodiag)[pairs[,1]],
         b = rownames(A_nodiag)[pairs[,2]])
}

direct_codep

a,b
<chr>,<chr>


In [7]:
# Verificación de que el grafo es un DAG (no tiene ciclos)
is_dag(g)

[1] TRUE

In [8]:
# Revisar componentes fuertemente conexas (ciclos)
scc <- components(g, mode = "strong")
groups <- split(V(g)$name, scc$membership)

problematic_scc <- groups[scc$csize > 1]
problematic_scc

named list()

In [9]:
## Orden topológico (solo si es un DAG) - orden de aprendizaje sin violar precedencias (podría no ser único)
if (is_dag(g)) {
  topo <- topo_sort(g, mode = "out")
  as.character(topo)
}

[1] "1"  "16" "17" "2"  "31" "35" "15" "30" "3"  "4"  "32" "5"  "24" "36" "6" 
[16] "18" "25" "7"  "33" "8"  "9"  "19" "34" "10" "20" "21" "26" "28" "22" "27"
[31] "11" "29" "23" "12" "13" "14"

Hasta aquí verificamos que la matriz que hay en Matriz_de_Adyacencia_Pensiones_v6.xlsx esta ok para ser usada

In [ ]:
# Identificar aristas redundantes: u->v es redundante si existe w con u->w y w->v (w != v)
stopifnot(is_dag(g))  

# lista de aristas (con nombres)
el <- as.data.frame(as_edgelist(g)) %>%
  setNames(c("from","to"))

# matriz de distancias / alcanzabilidad (reachability)
nodes <- V(g)$name
D <- distances(g, v = nodes, to = nodes, mode = "out")

# reach[u,v] = TRUE si hay camino u -> v (largo >= 1)
reach <- is.finite(D) & (D > 0)
dimnames(reach) <- list(nodes, nodes)

# redundante si existe w con u->w y w->v (w != v)
is_redundant <- logical(nrow(el))

for (k in seq_len(nrow(el))) {
  u <- el$from[k]
  v <- el$to[k]

  tmp <- reach[u, ] & reach[, v]
  tmp[v] <- FALSE   # excluye el caso w=v
  is_redundant[k] <- any(tmp)
}

redundant_edges <- el[is_redundant, , drop = FALSE] %>%
  arrange(from, to)

#redundant_edges

In [ ]:
## Eliminar aristas redundantes de la matriz de adyacencia
A_min <- A

idx <- cbind(redundant_edges$from, redundant_edges$to)
for (k in seq_len(nrow(idx))) {
  A_min[idx[k,1], idx[k,2]] <- 0
}

# reconstruir grafo y chequear que sigue siendo DAG
A_tmp <- A_min; diag(A_tmp) <- 0
g_min <- graph_from_adjacency_matrix(A_tmp, mode="directed", diag=FALSE)

is_dag(g_min)

In [ ]:
## Verificar que el grafo reducido tiene la misma alcanzabilidad (reachability) que el original
nodes <- V(g)$name

R0 <- distances(g,     v=nodes, to=nodes, mode="out")
R1 <- distances(g_min, v=nodes, to=nodes, mode="out")

reach0 <- is.finite(R0) & R0 > 0
reach1 <- is.finite(R1) & R1 > 0

all(reach0 == reach1)

In [ ]:
## Verificación alternativa: para cada arco eliminado u->v, verificar que no existe w con u->w y w->v (lo que haría que u->v sea redundante)
# debería dar FALSE
any({
  el_min <- as.data.frame(as_edgelist(g_min)); colnames(el_min) <- c("from","to")
  nodes <- V(g_min)$name
  D <- distances(g_min, v=nodes, to=nodes, mode="out")
  reach <- is.finite(D) & D > 0; dimnames(reach) <- list(nodes, nodes)
  sapply(seq_len(nrow(el_min)), function(k){
    u <- el_min$from[k]; v <- el_min$to[k]
    tmp <- reach[u,] & reach[,v]; tmp[v] <- FALSE
    any(tmp)
  })
})

In [ ]:
# 1) pasar matriz -> df, conservando nombres de fila
A_min_df <- A_min %>%
  as.data.frame() %>%
  rownames_to_column(var = "concepto")  # primera col = nombre del concepto (filas)

# 2) guardar a Excel (una hoja)
write_xlsx(
  x = list(A_min = A_min_df),
  path = "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Intermedias\\A_min.xlsx"
)

Ahora vamos a transformar esta matriz A_min a una forma que pueda ser leída en el modelo

In [10]:
idx <- which(A == 1, arr.ind = TRUE)

precedence_names <- tibble(
  i = rownames(A)[idx[,1]],
  j = colnames(A)[idx[,2]]
)

#precedence_names

In [11]:
path <- "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Intermedias\\instance_template.xlsx"
concepts_df <- read_excel(path, sheet = "concepts", col_names = TRUE)

In [12]:
lookup <- concepts_df %>% select(id, name)

precedence_ids <- precedence_names %>%
  left_join(lookup, by = c("i" = "name")) %>% rename(i_id = id) %>%
  left_join(lookup, by = c("j" = "name")) %>% rename(j_id = id) %>%
  select(i = i_id, j = j_id)

precedence_ids

i,j
<dbl>,<dbl>
1,2
2,3
15,3
2,4
15,4
3,5
3,6
4,6
15,6


In [13]:
library(writexl)

write_xlsx(list(precedence = precedence_ids), "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Intermedias\\precedence_completa.xlsx")